# Assignment 7B – K-Medoids Clustering on Credit Card Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn_extra.cluster import KMedoids

In [ ]:
df = pd.read_csv('credit_card.csv')
print(df.shape)
df.head(3)

## 1. Clean – Handle Missing Values

In [ ]:
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df = df.drop(columns=['CUST_ID'])
df['MINIMUM_PAYMENTS'] = df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].median())
df['CREDIT_LIMIT']     = df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median())
print('\nMissing after fill:', df.isnull().sum().sum())

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

## 2. Silhouette Method – Determine Optimal k

In [ ]:
# Sample for speed (KMedoids is O(n²))
np.random.seed(42)
idx = np.random.choice(len(X_scaled), 1000, replace=False)
X_sample = X_scaled[idx]

sil_scores = []
k_range    = range(2, 9)

for k in k_range:
    km = KMedoids(n_clusters=k, random_state=42)
    labels = km.fit_predict(X_sample)
    sil_scores.append(silhouette_score(X_sample, labels))

best_k = k_range[np.argmax(sil_scores)]
print(f'Best k by Silhouette: {best_k}')

plt.figure(figsize=(7, 4))
plt.plot(list(k_range), sil_scores, marker='o')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Method – Credit Card')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Fit K-Medoids

In [ ]:
km = KMedoids(n_clusters=best_k, random_state=42)
df['Cluster'] = km.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df['Cluster'])
print(f'Silhouette Score (full data, k={best_k}): {sil:.4f}')

In [ ]:
# Cluster sizes
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Cluster profiles – mean of key features
profile = df.groupby('Cluster')[['BALANCE','PURCHASES','CREDIT_LIMIT','PAYMENTS']].mean()
print(profile)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Balance vs Purchases scatter
colours = ['tomato','steelblue','seagreen','orchid','gold'][:best_k]
for c, col in zip(range(best_k), colours):
    mask = df['Cluster'] == c
    axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 2],
                    c=col, label=f'Cluster {c}', alpha=0.4, s=10)
axes[0].set_title('K-Medoids Clusters (Balance vs Credit Limit)')
axes[0].set_xlabel('Balance (scaled)')
axes[0].set_ylabel('Credit Limit (scaled)')
axes[0].legend()

# Cluster size bar
cluster_counts = df['Cluster'].value_counts().sort_index()
axes[1].bar([f'Cluster {i}' for i in cluster_counts.index],
            cluster_counts.values, color=colours)
axes[1].set_title('Cluster Sizes')
axes[1].set_ylabel('Count')

plt.suptitle('K-Medoids Clustering – Credit Card Dataset')
plt.tight_layout()
plt.show()